In [0]:
%sql
SELECT * FROM workspace.electiva.customers LIMIT 5;

SE CREAN LAS 8 TABLAS 

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.electiva.customers
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/customers.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.employee
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/employees.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.orders
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/orders.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_categories
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/productcategories.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/productsubcategories.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.products
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/products.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/vendors.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor_product
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/mi_volumen_datos/vendorproduct.csv', format => 'csv', header => true, inferSchema => true);

In [0]:
%sql

SELECT
    FirstName,
    LEFT(FirstName, 2) AS dos_letras
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT
    FirstName,
    UPPER(LEFT(FirstName, 2)) AS dos_letras
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT
    LastName,
    LOWER(LEFT(LastName, 2)) AS dos_letras_apellido
FROM workspace.electiva.customers;

1. PUNTO 

In [0]:
%sql
SELECT
    CustomerID AS customerid,
    FullName AS fullname,
    CONCAT(
        CustomerID, '-',
        UPPER(LEFT(FirstName, 2)), '-',
        LOWER(LEFT(LastName, 2)), '-',
        'AW'
    ) AS user_name
FROM workspace.electiva.customers;

2.PUNTO 

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.electiva.page_user (
    customerid INT,
    user_name STRING
) USING DELTA;

In [0]:
%sql
INSERT OVERWRITE workspace.electiva.page_user
SELECT
    CustomerID AS customerid,
    CONCAT(
        CustomerID, '-',
        UPPER(LEFT(FirstName, 2)), '-',
        LOWER(LEFT(LastName, 2)), '-',
        'AW'
    ) AS user_name
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT * FROM workspace.electiva.page_user;

In [0]:
%sql
SELECT
    SalesOrderID,
    OrderDate,
    to_date(OrderDate, 'M/d/yyyy') AS fecha
FROM workspace.electiva.orders
LIMIT 10;

In [0]:
%sql
SELECT
    SalesOrderID,
    OrderDate,
    to_date(OrderDate, 'M/d/yyyy') AS fecha
FROM workspace.electiva.orders
WHERE month(to_date(OrderDate, 'M/d/yyyy')) = 8
  AND year(to_date(OrderDate, 'M/d/yyyy')) = 2011;

3. PUNTO 

In [0]:
%sql
SELECT
    p.ProductName AS productname,
    p.ModelName AS modelname,
    o.SalesOrderID AS salesorderid,
    c.CategoryName AS categoryname,
    to_date(o.OrderDate, 'M/d/yyyy') AS orderdate,
    o.LineTotal AS linetotal
FROM workspace.electiva.orders AS o
JOIN workspace.electiva.products AS p
    ON o.ProductID = p.ProductID
JOIN workspace.electiva.product_sub_categories AS s
    ON p.SubCategoryID = s.SubCategoryID
JOIN workspace.electiva.product_categories AS c
    ON s.CategoryID = c.CategoryID
WHERE month(to_date(o.OrderDate, 'M/d/yyyy')) = 8
  AND year(to_date(o.OrderDate, 'M/d/yyyy')) = 2011;

In [0]:
%sql
SELECT
    p.ProductName AS productname,
    p.ModelName AS modelname,
    u.user_name AS user_name,
    o.SalesOrderID AS salesorderid,
    c.CategoryName AS categoryname,
    to_date(o.OrderDate, 'M/d/yyyy') AS orderdate,
    o.LineTotal AS linetotal,
    CASE
        WHEN c.CategoryName = 'Bikes' THEN 0.10
        WHEN c.CategoryName = 'Clothing' THEN 0.15
        WHEN c.CategoryName = 'Components' THEN 0.08
        WHEN c.CategoryName = 'Accessories' THEN 0.20
        ELSE 0
    END AS `descuento_mitad_año`
FROM workspace.electiva.orders AS o
JOIN workspace.electiva.products AS p
    ON o.ProductID = p.ProductID
JOIN workspace.electiva.product_sub_categories AS s
    ON p.SubCategoryID = s.SubCategoryID
JOIN workspace.electiva.product_categories AS c
    ON s.CategoryID = c.CategoryID
JOIN workspace.electiva.page_user AS u
    ON o.CustomerID = u.customerid
WHERE month(to_date(o.OrderDate, 'M/d/yyyy')) = 8
  AND year(to_date(o.OrderDate, 'M/d/yyyy')) = 2011;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.v_util_ordenes_mitad_anio AS
WITH base AS (
    SELECT
        p.ProductName AS productname,
        p.ModelName AS modelname,
        u.user_name AS user_name,
        o.SalesOrderID AS salesorderid,
        c.CategoryName AS categoryname,
        to_date(o.OrderDate, 'M/d/yyyy') AS orderdate,
        o.LineTotal AS linetotal,
        CASE
            WHEN c.CategoryName = 'Bikes' THEN 0.10
            WHEN c.CategoryName = 'Clothing' THEN 0.15
            WHEN c.CategoryName = 'Components' THEN 0.08
            WHEN c.CategoryName = 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
    FROM workspace.electiva.orders AS o
    JOIN workspace.electiva.products AS p
        ON o.ProductID = p.ProductID
    JOIN workspace.electiva.product_sub_categories AS s
        ON p.SubCategoryID = s.SubCategoryID
    JOIN workspace.electiva.product_categories AS c
        ON s.CategoryID = c.CategoryID
    JOIN workspace.electiva.page_user AS u
        ON o.CustomerID = u.customerid
    WHERE month(to_date(o.OrderDate, 'M/d/yyyy')) = 8
      AND year(to_date(o.OrderDate, 'M/d/yyyy')) = 2011
)
SELECT
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    `descuento_mitad_año`,
    linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM base;

In [0]:
%sql
SELECT * FROM workspace.electiva.v_util_ordenes_mitad_anio;

In [0]:
%sql
SELECT *
FROM workspace.electiva.v_util_ordenes_mitad_anio
ORDER BY user_name, salesorderid;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.ordenes_agosto_2011
USING DELTA AS
SELECT * FROM workspace.electiva.v_util_ordenes_mitad_anio;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.electiva.ordenes_agosto_2011;

4. PUNTO 

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
SELECT
    COUNT(DISTINCT o.SalesOrderID) AS amount_of_orders,
    u.user_name AS user_name,
    c.FullName AS customerfullname,
    CASE
        WHEN COUNT(DISTINCT o.SalesOrderID) >= 3 THEN 'Oro'
        WHEN COUNT(DISTINCT o.SalesOrderID) = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,
    o.CustomerID AS customerid
FROM workspace.electiva.orders AS o
JOIN workspace.electiva.customers AS c
    ON o.CustomerID = c.CustomerID
JOIN workspace.electiva.page_user AS u
    ON o.CustomerID = u.customerid
WHERE year(to_date(o.OrderDate, 'M/d/yyyy')) = 2011
GROUP BY o.CustomerID, c.FullName, u.user_name;

In [0]:
%sql
SELECT * FROM workspace.electiva.customer_class;

In [0]:
%sql
SELECT * FROM workspace.electiva.customer_class
ORDER BY customerid;

In [0]:
%sql
SELECT * FROM workspace.electiva.customer_class
WHERE customerid = 1043;